# SigAlg's `L2` class

In [1]:
# If running in Google Colab, uncomment the line below and run this cell first.
# Also, for Mac+Chrome users, beware of a known bug with LaTeX redering in Colab: https://github.com/googlecolab/colabtools/issues/3192

# !pip install sigalg

The `L2` class in SigAlg represents the *$L^2$-space* of square-integrable random variables on a probability space. The API reference is [here](https://johnmyers-phd.com/sigalg/api/modules/l2/#sigalg.l2.L2).

## Mathematical definition

Let $(\Omega, \mathcal{F}, P)$ be a probability space. The *$L^2$-space* associated with this probability space is the set

$$
L^2(\Omega, \mathcal{F}, P) = \left\{ X: \Omega \to \mathbb{R} \mid X \text{ is } \mathcal{F}\text{-measurable and } \int_\Omega X^2 \, dP < \infty \right\}.
$$

This is a vector space under pointwise addition and scalar multiplication. It becomes a *Hilbert space* when equipped with the inner product

$$
\langle X, Y \rangle \stackrel{\text{def}}{=} \int_\Omega XY \, dP = E(XY),
$$

which induces the *$L^2$-norm*

$$
\|X\| \stackrel{\text{def}}{=} \sqrt{\langle X, X \rangle} = \sqrt{E(X^2)},
$$

and the *$L^2$-metric*

$$
d(X,Y) \stackrel{\text{def}}{=} \|X - Y\| = \sqrt{E\left[(X-Y)^2\right]}.
$$

In the case that $\Omega$ is finite (as it always is in SigAlg), any $\mathcal{F}$-measurable random variable automatically satisfies $E(X^2) < \infty$, so $L^2(\Omega, \mathcal{F}, P)$ is simply the vector space of all $\mathcal{F}$-measurable random variables. Moreover, the space has an *orthonormal basis* consisting of the normalized indicator functions

$$
\phi_A \stackrel{\text{def}}{=} \frac{I_A}{\|I_A\|} = \frac{I_A}{\sqrt{P(A)}},
$$

where $A$ ranges over all atoms of $\mathcal{F}$ with nonzero probability. Any $X \in L^2(\Omega, \mathcal{F}, P)$ can thus be written as a *(generalized) Fourier series*

$$
X = \sum_{A} \langle X, \phi_A \rangle \phi_A,
$$

where the sum extends over all atoms with nonzero probability.

## API examples



### Creating $L^2$-spaces

#### From individual components

We begin by defining a sample space $\Omega = \{0,1,2,3,4,5\}$, a $\sigma$-algebra $\mathcal{F}$ on $\Omega$ with three atoms, and a probability measure $P$. 

In [2]:
from sigalg.core import ProbabilityMeasure, SampleSpace, SigmaAlgebra

Omega = SampleSpace().from_sequence(size=6)

F = SigmaAlgebra(sample_space=Omega, name="F").from_dict(
    {
        0: 0,  # atom with probability 0
        1: 0,  # atom with probability 0
        2: 1,
        3: 1,
        4: 1,
        5: 2,
    }
)

P = ProbabilityMeasure(sample_space=Omega).from_dict(
    {
        0: 0.0,
        1: 0.0,
        2: 0.45,
        3: 0.3,
        4: 0.2,
        5: 0.05,
    }
)

Create an instance of `L2` from these components.

In [3]:
from sigalg.l2 import L2

H = L2(sample_space=Omega, sig_alg=F, prob_measure=P, name="H")
print(H)

H = L2(Omega, F, P)

* Sample space 'Omega':
[0, 1, 2, 3, 4, 5]

* Sigma algebra 'F':
        atom ID
sample         
0             0
1             0
2             1
3             1
4             1
5             2

* Probability measure 'P':
        probability
sample             
0              0.00
1              0.00
2              0.45
3              0.30
4              0.20
5              0.05


#### Using default components

Alternatively, we can create an instance of `L2` using only a sample space, and letting SigAlg generate the power-set $\sigma$-algebra and the uniform probability measure.

In [4]:
K = L2(sample_space=Omega, name="K")
print(K)

K = L2(Omega, power_set, P)

* Sample space 'Omega':
[0, 1, 2, 3, 4, 5]

* Sigma algebra 'power_set':
        atom ID
sample         
0             0
1             1
2             2
3             3
4             4
5             5

* Probability measure 'P':
        probability
sample             
0          0.166667
1          0.166667
2          0.166667
3          0.166667
4          0.166667
5          0.166667


### Bases

The `dim` property of an `L2` instance gives the dimension of the space, which equals the number of atoms of the $\sigma$-algebra with nonzero probability.

In [5]:
print(f"Dimension of H: {H.dim}")
print(f"Dimension of K: {K.dim}")

Dimension of H: 2
Dimension of K: 6


Though the $\sigma$-algebra of `H` has three atoms, notice that its dimension is only $2$, because the atom $A_0 = \{0,1\}$ has zero probability.

The `basis` property returns a dictionary containing the orthonormal basis vectors $\phi_A = I_A / \|I_A\|$ for each atom $A$ with nonzero probability.

In [6]:
basis = H.basis
print("Orthonormal basis:")
for atom_id, phi in basis.items():
    print(f"  Atom {atom_id}: {phi}")

Orthonormal basis:
  Atom 1: Random variable '1':
               1
sample          
0       0.000000
1       0.000000
2       1.025978
3       1.025978
4       1.025978
5       0.000000
  Atom 2: Random variable '2':
               2
sample          
0       0.000000
1       0.000000
2       0.000000
3       0.000000
4       0.000000
5       4.472136


We can verify that these basis vectors are orthonormal by computing their inner products using the `inner` method. For details on inner products, see the dedicated [`inner` notebook](inner.ipynb).

In [7]:
print("Verifying orthonormality:")
for i, phi_i in basis.items():
    for j, phi_j in basis.items():
        inner_prod = H.inner(phi_i, phi_j)
        print(f"  <phi_{i}, phi_{j}>⟩ = {inner_prod:.6f}")

Verifying orthonormality:
  <phi_1, phi_1>⟩ = 1.000000
  <phi_1, phi_2>⟩ = 0.000000
  <phi_2, phi_1>⟩ = 0.000000
  <phi_2, phi_2>⟩ = 1.000000


The inner products are $1$ when $i=j$ (norm equals 1) and $0$ when $i \neq j$ (orthogonality), confirming the basis is orthonormal.

### Containment checking and almost-sure equality

We can check if a random variable is in the $L^2$-space using the `in` operator. A random variable is in $L^2(\Omega, \mathcal{F}, P)$ if and only if it is $\mathcal{F}$-measurable.

In [8]:
from sigalg.core import RandomVariable

# X is constant on the atoms of F — thus it is F-measurable
X = RandomVariable(domain=Omega).from_dict(
    {
        0: 2,
        1: 2,
        2: 3,
        3: 3,
        4: 3,
        5: -1,
    }
)

# Y is not constant on the atoms of F — thus it is *not* F-measurable
Y = RandomVariable(domain=Omega, name="Y").from_dict(
    {
        0: 1,
        1: 1,
        2: 1,
        3: -1,
        4: -1,
        5: 5,
    }
)

print(f"X in H: {X in H}")
print(f"Y in H: {Y in H}")

X in H: True
Y in H: False


Equality of random variables in an $L^2$-space is *almost-sure equality*, meaning that two random variables $X$ and $Y$ are considered equal if $P(X \neq Y) = 0$. We can check for almost-sure equality using the `almost_surely_equal` method on the `L2` instance.

In [9]:
# Recall the atom A_0 = {0, 1} of F has zero probability

U = RandomVariable(domain=Omega, name="U").from_dict(
    {
        0: 1,  # Differs from X(0)=2
        1: 1,  # Differs from X(0)=2
        2: 3,  # Equal to X(2)=3
        3: 3,  # Equal to X(3)=3
        4: 3,  # Equal to X(4)=3
        5: -1,  # Equal to X(5)=-1
    }
)

V = RandomVariable(domain=Omega, name="V").from_dict(
    {
        0: 2,  # Equal to X(0)=2
        1: 2,  # Equal to X(1)=2
        2: 4,  # Differs from X(2)=3
        3: 4,  # Differs from X(3)=3
        4: 4,  # Differs from X(4)=3
        5: -1,  # Equal to X(5)=-1
    }
)

print(f"Are X and U almost-surely equal? {H.almost_surely_equal(X, U)}")
print(f"Are X and V almost-surely equal? {H.almost_surely_equal(X, V)}")

Are X and U almost-surely equal? True
Are X and V almost-surely equal? False


### Fourier coefficients

As we mentioned earlier, any random variable in an `L2` instance can be expressed as a Fourier series in terms of the orthonormal basis contained in the `basis` attribute. The `fourier_coefficients` method computes the coefficients of this series.

In [10]:
coeffs = H.fourier_coefficients(X)
basis = H.basis
print("Fourier coefficients:")
for atom_id, coeff in coeffs.items():
    print(f"  Atom {atom_id}: {coeff:.6f}")

Fourier coefficients:
  Atom 1: 2.924038
  Atom 2: -0.223607


We can reconstruct $X$ from its Fourier coefficients by computing $X = \sum_A \langle X, \phi_A \rangle \phi_A$.

In [11]:
X_reconstructed = sum(coeffs[i] * basis[i] for i in coeffs.keys()).with_name(
    "X_reconstructed"
)

# Original X
print(X, "\n")

# Reconstructed X
print(X_reconstructed)

print(f"\nAre they almost-surely equal? {H.almost_surely_equal(X, X_reconstructed)}")

Random variable 'X':
        X
sample   
0       2
1       2
2       3
3       3
4       3
5      -1 

Random variable 'X_reconstructed':
        X_reconstructed
sample                 
0                   0.0
1                   0.0
2                   3.0
3                   3.0
4                   3.0
5                  -1.0

Are they almost-surely equal? True


### Related methods

The `L2` class provides several key methods for working with the Hilbert space structure. Each has its own dedicated notebook with detailed examples:

- [`inner`](inner.ipynb) - Compute the inner product $\langle X, Y \rangle = E(XY)$ of two random variables
- [`norm`](norm.ipynb) - Compute the $L^2$-norm $\|X\| = \sqrt{E(X^2)}$ of a random variable
- [`metric`](metric.ipynb) - Compute the $L^2$-distance $d(X,Y) = \|X-Y\|$ between random variables
- [`proj`](proj.ipynb) - Compute the orthogonal projection of a random variable onto a subspace

These methods form the foundation for understanding probabilistic concepts in terms of Hilbert space geometry.